# OpenShorts on Kaggle (2×T4)

**This notebook only gets the environment running.** Everything that could need
fixing lives in the repo (`kaggle_bootstrap.sh`, `kaggle_smoke_test.py`), so
improvements arrive with a `git pull` in cell 2 — you should not have to
re-import this notebook again.

Run cells 1–4 in order, then open the public URL printed at the end of cell 3
and use the app normally. There is deliberately no test/render scaffolding
here — test with the real workflow.

**Before running — notebook settings (right panel):**

| setting | value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** (needs a phone-verified account) |

**Add-ons → Secrets** — create each one and **tick its checkbox** for this
notebook (saving alone does not attach it):

| secret | needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (PAT with Contents:Read) |
| `GEMINI_API_KEY` | the picker + context layer — **without it nothing is clipped** |
| `GEMINI_API_KEYS` | **extra Gemini keys, comma-separated — strongly recommended** (the picker + context rotate across them on 429/quota) |
| `ASSEMBLYAI_API_KEY` | transcription **with diarization** (without it, local whisper runs — slower and no diarization) |
| `HF_TOKEN` + `HF_STORAGE_REPO` | **write** token + `<user>/openshorts-clips` — clips survive the session and upload as each one finishes |

Optional — only needed if you use the feature:

| secret | feature |
|---|---|
| `CONTEXT_GEMINI_API_KEY` | dedicated key for the pre-download context layer (default: reuses `GEMINI_API_KEY`) |
| `YOUTUBE_COOKIES` | paste a working cookies.txt if downloads get blocked |
| `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_BUCKET` | S3 backup |
| `UPLOAD_POST_API_KEY` | auto-publish to TikTok/Instagram/YouTube |
| `ELEVENLABS_API_KEY` | voice dubbing |

Unknown/unset secrets are skipped silently, so you can add one later by
creating it in Kaggle — no notebook edit needed.


In [ ]:
# ── Cell 1 — secrets ───────────────────────────────────────────────────────
# The ONLY thing this notebook knows. Everything else lives in the repo.
#
# Names are loaded from Kaggle Secrets when present. To paste a key instead
# (quicker, but it is saved inside the notebook and its version history — a
# public fork or download takes the key with it), put it in PASTED below.

SECRETS = [
    "GITHUB_TOKEN",
    "GEMINI_API_KEY", "GEMINI_API_KEYS", "CONTEXT_GEMINI_API_KEY",
    "ASSEMBLYAI_API_KEY",
    "HF_TOKEN", "HF_STORAGE_REPO",
    "YOUTUBE_COOKIES",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_BUCKET",
    "UPLOAD_POST_API_KEY", "ELEVENLABS_API_KEY",
]

PASTED = {
    # "GEMINI_API_KEY": "...",
}

import os

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None

loaded = []
for name in SECRETS:
    value = (PASTED.get(name) or "").strip()
    source = "pasted"
    if not value and _secrets is not None:
        try:
            value, source = _secrets.get_secret(name).strip(), "secret"
        except Exception:
            value = ""
    if value:
        os.environ[name] = value
        loaded.append(name)
        print(f"  ok        {name:<26} ({source}, {len(value)} chars)")
    else:
        print(f"  not set   {name}")

if "GEMINI_API_KEY" not in loaded:
    print("\n  !! GEMINI_API_KEY is missing — no clips will be selected.")
if "GEMINI_API_KEYS" not in loaded:
    print("  !  No GEMINI_API_KEYS — the picker/context use only the primary")
    print("     key, so a quota hit can slow or fail a long job.")
if "ASSEMBLYAI_API_KEY" not in loaded:
    print("  !  No ASSEMBLYAI_API_KEY — local whisper will run: slower, and no")
    print("     diarization, which costs the framing policy an evidence tier.")
if "HF_TOKEN" not in loaded or "HF_STORAGE_REPO" not in loaded:
    print("  !  No HF storage — clips are wiped when this session ends.")


In [ ]:
# ── Cell 2 — get the code (clone the first time, pull after that) ─────────
import os, subprocess

BRANCH = "claude/gemini-vision-clip-picking-bikvuy"
DEST = "/kaggle/working/openshorts"
PRIVATE_REPO = True   # set False if you make the repo public

have_token = bool(os.environ.get("GITHUB_TOKEN"))
if PRIVATE_REPO and not have_token:
    raise SystemExit(
        "GITHUB_TOKEN not loaded.\n"
        "  -> Add-ons > Secrets: TICK THE CHECKBOX next to GITHUB_TOKEN\n"
        "     (saving the secret is not enough; it must be attached to this\n"
        "     notebook), then re-run cell 1 and this one.")

# Never printed: it carries the token.
url = (f"https://{os.environ['GITHUB_TOKEN']}@github.com/foskigr8/openshorts.git"
       if have_token else "https://github.com/foskigr8/openshorts.git")

if os.path.isdir(os.path.join(DEST, ".git")):
    # Idempotent: re-running this cell picks up new commits instead of
    # reporting "already cloned" and leaving you on stale code.
    subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", url],
                   capture_output=True, text=True)
    r = subprocess.run(["git", "-C", DEST, "pull", "--ff-only", "origin", BRANCH],
                       capture_output=True, text=True)
    print("pull ok" if r.returncode == 0 else
          f"PULL FAILED (continuing on the existing checkout):\n{r.stderr[-400:]}")
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr[-400:]
        hint = ""
        if "could not read Username" in err:
            hint = "\n  -> the token was not applied; see the checkbox note above"
        elif "Authentication failed" in err or "403" in err:
            hint = ("\n  -> token rejected: check it has Contents:Read on"
                    " foskigr8/openshorts and has not expired")
        elif "Remote branch" in err:
            hint = f"\n  -> branch '{BRANCH}' not found on the remote"
        raise SystemExit(f"CLONE FAILED:\n{err}{hint}")
    print("clone ok")

os.chdir(DEST)
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 3 — install, build, serve, tunnel (5-12 min on a cold session) ──
# Ends by printing a public URL. Re-run with SKIP_INSTALL=1 to skip pip/npm.
# First run installs onnxruntime-gpu (~2GB) so face spine runs on the GPU —
# Cell 3 must print 'onnxruntime-gpu installed: CUDAExecutionProvider
# available' and 'CUDA decode (NVDEC + CUDA filters) also present' before you
# run a job. Cell 4 verifies all of it.
!bash kaggle_bootstrap.sh


In [ ]:
# ── Cell 4 — is it actually working? ──────────────────────────────────────
# Starting is not the same as working. This checks each thing that breaks
# independently and names the one that failed. It lives in the repo, so it
# improves with a `git pull` — no notebook re-import.
!python3 kaggle_smoke_test.py


## Optional — keep the session alive

Jupyter runs **one cell at a time**, so a blocking `while True:` loop locks the
notebook. This runs the heartbeat on a background thread, so the cell returns
immediately.

You do not need it while actively clicking around; it matters when you walk
away mid-render or use *Save & Run All*.

**Stopping any cell is safe** — the backend and tunnel run under `nohup`.

**Clips survive the session only if HF storage is configured** (`HF_TOKEN` +
`HF_STORAGE_REPO` in cell 1). They upload as each clip finishes, and History
serves them from HF after `/kaggle/working` is wiped.


In [ ]:
import threading, time

def _heartbeat():
    # Touches a file rather than printing: notebook output from a background
    # thread interleaves with whatever cell you are running next, which makes
    # the notebook unreadable.
    while True:
        with open('/kaggle/working/.heartbeat', 'w') as f:
            f.write(str(time.time()))
        time.sleep(60)

if not any(t.name == 'openshorts-heartbeat' for t in threading.enumerate()):
    threading.Thread(target=_heartbeat, name='openshorts-heartbeat',
                     daemon=True).start()
    print('heartbeat started on a background thread — this cell is free')
else:
    print('heartbeat already running')
